This is the code for **Homework 5: Consistent Personas** for CMSC 412/612: Neurosymbolic Text Generation.

You will use the PeaCoK persona knowledge graph ([Gao et al., ACL 2023](https://aclanthology.org/2023.acl-long.362/)) to:

* **Task A (12 pts):** build interconnected personas (Q A.1–A.3).
* **Task B (9 pts):** test whether PeaCoK knowledge helps an LLM stay consistent (Q B.1–B.4).

**Everything here is free:** no API keys. It runs on CPU, but a free GPU is much faster (Colab: *Runtime → Change runtime type → T4 GPU*).

**How to work through it:** fill in every block marked `# TODO`, running cells top to bottom. Don't rename functions or change their signatures. Each question says what to **Do** (code), **Report** (for your PDF), and **Answer** (written, for your PDF).

**Submit three things:**
1) This ipynb notebook that has your code and outputs saved.
2) A single PDF copy with the answers that need to be reported
3) The dialogues.json file you generate in Task B

# Setup

In [ ]:
!pip -q install -U sentence-transformers gdown "huggingface_hub<2.0" networkx

In [ ]:
# TODO: set this to the last four digits of your student ID (as a string, e.g. "0412").
# We're going to use it as a seed number for this assignment.
STUDENT_ID_LAST4 =

In [ ]:
# Provided code
assert STUDENT_ID_LAST4.isdigit() and len(STUDENT_ID_LAST4) == 4, "Use the last four digits of your student ID."
assert STUDENT_ID_LAST4 != "0000", "Set STUDENT_ID_LAST4 to the last four digits of your student ID."
SEED = int(STUDENT_ID_LAST4)

In [ ]:
import os, re, json, random, string, zipfile, itertools, collections, glob, gc
import numpy as np
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt
import torch
from getpass import getpass

def set_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s)
set_seed(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
# Ask for a Hugging Face token (hidden as you type, so it is not saved in the notebook).
# Create one at https://huggingface.co/settings/tokens (read access is enough).
HF_TOKEN = getpass("Enter your HF_TOKEN: ").strip()
assert HF_TOKEN, "Enter a Hugging Face token to continue."
os.environ["HF_TOKEN"] = HF_TOKEN  # every Hugging Face download below picks this up automatically

# Free, open-weight models.
GEN_MODEL_NAME   = "Qwen/Qwen2.5-1.5B-Instruct" if DEVICE == "cuda" else "Qwen/Qwen2.5-0.5B-Instruct"
EMB_MODEL_NAME   = "sentence-transformers/all-MiniLM-L6-v2"
NLI_MODEL_NAME   = "cross-encoder/nli-deberta-v3-base"
print("Seed:", SEED, "| device:", DEVICE, "| generator:", GEN_MODEL_NAME)

In [ ]:
# Provided: load a chat model and talk to it. Used for generation (Task A, Task B).
from transformers import AutoTokenizer, AutoModelForCausalLM

def load_chat_model(name):
    tok = AutoTokenizer.from_pretrained(name)
    dtype = torch.float16 if DEVICE == "cuda" else torch.float32
    try:
        model = AutoModelForCausalLM.from_pretrained(name, dtype=dtype)
    except TypeError:                     # older transformers versions
        model = AutoModelForCausalLM.from_pretrained(name, torch_dtype=dtype)
    return tok, model.to(DEVICE).eval()

def chat_with(tok, model, messages, max_new_tokens=96, temperature=0.7, do_sample=True):
    prompt = tok.apply_chat_template(messages, add_generation_prompt=True, tokenize=False)
    inputs = tok(prompt, return_tensors="pt").to(DEVICE)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=do_sample,
                             temperature=temperature if do_sample else None,
                             top_p=0.9 if do_sample else None,
                             pad_token_id=tok.pad_token_id if tok.pad_token_id is not None else tok.eos_token_id)
    return tok.decode(out[0, inputs["input_ids"].shape[-1]:], skip_special_tokens=True).strip()

gen_tok, gen_model = load_chat_model(GEN_MODEL_NAME)
def chat(messages, **kw):
    return chat_with(gen_tok, gen_model, messages, **kw)

print(chat([{"role": "user", "content": "Say hello in five words."}], do_sample=False))

config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 3.09GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Hello there! How can I assist you today?


# Task A: Generate Interconnected Personas (12 pts)

## Provided: load PeaCoK and build the persona graph

The full graph (`peacok_kg.json`, about 171 MB, license CC BY-NC 4.0) is downloaded from the authors' [Hugging Face page](https://huggingface.co/datasets/Silin1590/PeaCoK), with their Google Drive copy as a backup. If both fail, a tiny **demo sample** is used so you can test your code, but your write-up must use the real graph.

Each fact is a triple (head persona, relation, tail attribute) with three labels: a main relation (`characteristic`, `routine_habit`, `goal_plan`, `experience`), interactivity (`relationship` / `self`), and distinctiveness (`distinctive` / `generic`). The cells below keep only valid distinctive facts and build a graph that links each persona to its attributes. You don't need to edit them.

In [ ]:
# Provided code
# Fetch the PeaCoK knowledge graph from Hugging Face (dataset repo Silin1590/PeaCoK).
# The repo holds one raw file, peacok_kg.json: a nested dict keyed by persona, not a table,
# so `datasets.load_dataset` cannot parse it. hf_hub_download fetches the file as-is
# (cached after the first run; uses HF_TOKEN from the environment if set).
from huggingface_hub import hf_hub_download

def download_peacok():
    path = hf_hub_download(repo_id="Silin1590/PeaCoK", filename="peacok_kg.json", repo_type="dataset")
    assert os.path.exists(path) and os.path.getsize(path) > 1_000_000, f"PeaCoK download looks incomplete: {path}"
    return path

def verify_peacok(path):
    # Open and parse the whole file: proves it was fully downloaded and is the expected JSON graph.
    with open(path, encoding="utf-8") as f:
        data = json.load(f)                       # raises JSONDecodeError if the file is truncated/corrupt
    assert isinstance(data, dict) and data, "PeaCoK file parsed, but it is not a non-empty dict of personas."
    first = next(iter(data.values()))
    assert "attributes" in first, "PeaCoK file has an unexpected layout (no 'attributes' key)."
    return len(data), sum(len(info["attributes"]) for info in data.values())

PEACOK_FILE = download_peacok()
n_personas, n_facts = verify_peacok(PEACOK_FILE)
print("PeaCoK file:", PEACOK_FILE, f"({os.path.getsize(PEACOK_FILE) / 1e6:.0f} MB)")
print(f"Verified: parsed OK, {n_personas} personas, {n_facts} persona-attribute facts")

peacok_kg.json: reconstructing file:   0%|          |  0.00B /  171MB            

peacok_kg.json: downloading bytes:           |  0.00B            

PeaCoK file: /root/.cache/huggingface/hub/datasets--Silin1590--PeaCoK/snapshots/1815b92d8255ecb906e423b4be54d2d0adcd5cb4/peacok_kg.json (171 MB)
Verified: parsed OK, 3774 personas, 115094 persona-attribute facts


In [ ]:
# Provided code
def load_peacok(path):
    # peacok_kg.json layout (Silin1590/PeaCoK on Hugging Face):
    # {head: {"attributes": {tail: {"relation_labels": {"majority": [main, interactivity, distinctiveness]}, ...}}}}
    # "majority" is the label most annotators agreed on; heads are lowercase first-person sentences ("i am a nurse").
    assert path and os.path.exists(path), "PeaCoK file not found; re-run the download cell above."
    with open(path, encoding="utf-8") as f:
        data = json.load(f)
    rows = []
    for head, info in data.items():
        for tail, a in info["attributes"].items():
            main, interactivity, distinctiveness = a["relation_labels"]["majority"]
            rows.append((head.strip(), tail.strip(), main, interactivity, distinctiveness))
    return pd.DataFrame(rows, columns=["head", "tail", "main", "interactivity", "distinctiveness"])

VALID_MAIN = {"characteristic", "routine_habit", "goal_plan", "experience"}
VALID_INTERACTIVITY = {"relationship", "self"}

# Table 19 of the paper: how a fact is written as a sentence
REL_TEXT = {"characteristic": "here is my character trait",
            "routine_habit": "here is what I regularly or consistently do",
            "goal_plan": "here is what I will do or achieve in the future",
            "experience": "here is what I did in the past",
            "relationship": "related to other people or social groups"}

def filter_facts(df):
    # Keep valid, distinctive persona facts (the paper drops generic facts before linking them to dialogues).
    # This also drops the "controversial" and "not" main labels, which are not persona relations.
    mask = (df["main"].isin(VALID_MAIN) & df["interactivity"].isin(VALID_INTERACTIVITY)
            & (df["distinctiveness"] == "distinctive"))
    return df[mask].reset_index(drop=True)

def fact_to_statement(head, main, interactivity, tail):
    desc = REL_TEXT[main] + (" " + REL_TEXT["relationship"] if interactivity == "relationship" else "")
    return f"{head.strip().rstrip('.')}, {desc}, {tail.strip().rstrip('.')}."

def _norm_attr(t):
    return re.sub(r"\s+", " ", str(t).lower().strip().rstrip("."))

def build_persona_graph(facts):
    # Bipartite graph: persona nodes ("P", head) and attribute nodes ("A", tail); one edge per fact.
    # The "P"/"A" tags matter: a few hundred attribute strings are identical to some persona string.
    G = nx.Graph()
    for r in facts.itertuples(index=False):
        p, a = ("P", r.head), ("A", _norm_attr(r.tail))
        G.add_node(p, node_type="persona"); G.add_node(a, node_type="attribute")
        G.add_edge(p, a, relation=r.main, interactivity=r.interactivity)
    return G

peacok_df = load_peacok(PEACOK_FILE)
for col in ["main", "interactivity", "distinctiveness"]:
    print(col, peacok_df[col].value_counts().to_dict())     # check the label spellings match the sets above
facts = filter_facts(peacok_df)
G = build_persona_graph(facts)
print(len(facts), "distinctive facts |", facts["head"].nunique(), "personas |", G.number_of_edges(), "edges")

main {'routine_habit': 47375, 'characteristic': 22093, 'experience': 17630, 'goal_plan': 14770, 'controversial': 10902, 'not': 2324}
interactivity {'self': 87645, 'relationship': 27449}
distinctiveness {'distinctive': 90267, 'generic': 24827}
82965 distinctive facts | 3774 personas | 82961 edges


In [ ]:
# Provided code
# Look at the data: 5 random personas with all their distinctive attributes, grouped by relation type.
# "[rel]" marks attributes that involve other people.
PEEK_LABEL = {"characteristic": "character trait", "routine_habit": "routine / habit",
              "goal_plan": "goal / plan", "experience": "experience"}
MAX_PER_REL = 6   # cap per relation type so the output stays readable

rng = random.Random(SEED)
sample_personas = rng.sample(sorted(facts["head"].unique()), 5)
for head in sample_personas:
    sub = facts[facts["head"] == head]
    print(f"\n== {head}  ({len(sub)} distinctive attributes)")
    for rel in PEEK_LABEL:
        rows = sub[sub["main"] == rel]
        if rows.empty:
            continue
        items = [t + (" [rel]" if inter == "relationship" else "") for t, inter in zip(rows["tail"], rows["interactivity"])]
        more = f" (+{len(items) - MAX_PER_REL} more)" if len(items) > MAX_PER_REL else ""
        print(f"  {PEEK_LABEL[rel]:>15}: {'; '.join(items[:MAX_PER_REL])}{more}")


== i am a celebrity who am a star athlete  (25 distinctive attributes)
  character trait: dominant; have a lot of fans [rel]; be a winner; victorious; perseverance and determination; physically active and fit (+1 more)
  routine / habit: wins a game; practice a sport; train consistently; win the competition; practices with a sports team [rel]; participates in sporting events (+4 more)
      goal / plan: i get a scholarship; try out for professional league [rel]; win a championship; to break a world record; to create a charitable foundation [rel]
       experience: won a championship title; won multiple sports championships; won a major sports championship

== i am a scrabble player  (29 distinctive attributes)
  character trait: good at word games; intelligent; enjoys word games; smart
  routine / habit: spread pieces out; win game; play game; play with friends [rel]; skilled at making words from letters; studies words (+8 more)
      goal / plan: to make words from random letters; to

## Q A.1 (4 pts): Find bridges between personas

Two personas are connected by a **bridge** when they share a distinctive attribute (paper Sec. 5). For example, *study music at college* is an `experience` for a singer but a `routine_habit` for a music student.

* **Do:** implement `find_bridges(G, persona_a, persona_b)` and `most_connected_personas(G, seed, k)`.
* **Report:** your seed persona, its 5 most-connected personas, and the bridges for the best pair.

In [ ]:
# Provided code
def random_persona(G, rng):
    # Pick one persona node at random. Sorted first so the same rng seed always gives the same persona.
    return rng.choice(sorted(n[1] for n in G if n[0] == "P"))

In [ ]:
# TODO
def most_connected_personas(G, seed, min_shared=3):
    # TODO: return [(persona, n_shared_attributes), ...] for every persona that shares at least `min_shared`
    #       attributes with `seed`, most shared first; break ties alphabetically. Do not include `seed` itself.
    #
    # How the graph is laid out (see build_persona_graph): persona nodes are ("P", head), attribute nodes are
    # ("A", tail), and every edge joins one persona to one attribute. So another persona shares an attribute
    # with `seed` exactly when it is two hops away:  seed -> attribute -> other persona.
    #
    # Hints:
    #   1. Start from the seed's node, s = ("P", seed).
    #   2. For each attribute in G.neighbors(s), look at G.neighbors(attribute): those are the personas that
    #      have that attribute. Each time you meet a persona other than s, it shares one more attribute with
    #      the seed, so add 1 to its count (a collections.Counter keyed by the head text, node[1], works well).
    #   3. Keep only the personas whose count is >= min_shared.

    # YOUR CODE HERE


def find_bridges(G, persona_a, persona_b):
    # TODO: return a list of dicts, one per attribute that BOTH personas have, sorted by attribute text:
    #       {"attribute": str, "relation_a": str, "relation_b": str, "same_relation": bool}
    #       relation_a / relation_b say how each persona relates to that attribute (e.g. "experience" for one,
    #       "routine_habit" for the other, as in the singer / music-student example); same_relation is
    #       relation_a == relation_b.
    #
    # Hints:
    #   1. The two persona nodes are a = ("P", persona_a) and b = ("P", persona_b).
    #   2. A persona's attributes are its neighbours, set(G.neighbors(a)). A bridge is an attribute node that is
    #      in BOTH neighbour sets, so intersect the two sets with &.
    #   3. The relation is stored on the edge, not on the node: G.edges[a, n]["relation"] is how persona a relates
    #      to attribute n (build_persona_graph sets relation=r.main on every edge).

    # YOUR CODE HERE

    return bridges

In [ ]:
# Provided code
MIN_SHARED = 3   # a persona counts as connected if it shares at least this many attributes with the seed

# Step 1: a random seed persona (seeded with SEED). Redraw until it has at least one connected persona.
rng = random.Random(SEED)
top = []
while not top:
    SEED_PERSONA = random_persona(G, rng)
    top = most_connected_personas(G, SEED_PERSONA, min_shared=MIN_SHARED)
print("Seed persona:", SEED_PERSONA)
print(f"Top 2 personas sharing >= {MIN_SHARED} attributes with it:")
display(pd.DataFrame(top[:2], columns=["persona", "n_shared_attributes"]))

# Step 2: the bridges between the seed and its most-connected persona.
best, n_best = top[0]
print(f"\nBridges between '{SEED_PERSONA}' and '{best}' ({n_best} shared attributes):")
display(pd.DataFrame(find_bridges(G, SEED_PERSONA, best)))

Seed persona: i am a celebrity who am a star athlete
Top 2 personas sharing >= 3 attributes with it:


,persona,n_shared_attributes
0,i am a football player who am a star athlete,11
1,i am a professional basketball player who am a...,11



Bridges between 'i am a celebrity who am a star athlete' and 'i am a football player who am a star athlete' (11 shared attributes):


,attribute,relation_a,relation_b,same_relation
0,be a winner,characteristic,characteristic,True
1,dominant,characteristic,characteristic,True
2,have a lot of fans,characteristic,characteristic,True
3,i get a scholarship,goal_plan,goal_plan,True
4,practice a sport,routine_habit,routine_habit,True
5,train consistently,routine_habit,routine_habit,True
6,try out for professional league,goal_plan,goal_plan,True
7,victorious,characteristic,characteristic,True
8,win a championship,goal_plan,goal_plan,True
9,win the competition,routine_habit,goal_plan,False


## Q A.2 (6 pts): Generate persona frames for three original characters

PeaCoK's authors generated persona attributes by prompting InstructGPT-3 with an instruction and 5 examples per relation (Instruction copied from the paper into `FEW_SHOT` below). You will do the same with a free local model.

* **Do:** write three original characters as PeaCoK-style personas (*"I am a ... who ..."*). Implement `build_attribute_prompt()` and `parse_attributes()` so the model generates attributes for all five relations.
* **Report:** the generated frames for all three characters, and (from the provided cell) the retrieved PeaCoK frame for one character.
* **Answer (3–4 sentences):** comparing the generated and retrieved frames for that character, which is more distinctive and which is more plausible? Did the model produce anything stereotyped or contradictory?

In [ ]:
# TODO: replace with YOUR three original characters (name -> persona). No existing fictional characters or real people.
# Example - "Theo": "I am a retired lighthouse keeper who restores old wooden boats"

MY_CAST = {
    # YOUR CODE HERE
}

In [ ]:
# Provided code
assert len(MY_CAST) == 3 and all("..." not in p for p in MY_CAST.values()), "Write your three characters."

RELATIONS = ["characteristic", "routine_habit", "goal_plan", "experience", "relationship"]
REL_LABEL = {"characteristic": "Characteristic", "routine_habit": "Routine or Habit", "goal_plan": "Goal or Plan",
             "experience": "Experience", "relationship": "Relationship"}

# Instructions and 'Simple Head Personas' examples from PeaCoK Tables 12-13 (Gao et al., 2023)
INSTRUCTIONS = {
    "characteristic": "Guess a character trait of the person in the clue, which can distinguish this person from others.",
    "routine_habit": "Guess what the person in the clue regularly or consistently does, which can distinguish this person from others.",
    "goal_plan": "Guess what the person in the clue will do or achieve in the future, which can distinguish this person from others.",
    "experience": "Guess what the person in the clue did in the past, which can distinguish this person from others.",
    "relationship": "Guess a relationship that the person in the clue has with other people or social groups, which can distinguish this person from others.",
}
CLUES = ["I become an accountant.", "I want to be an actor.", "I am an alert person.", "I work as a lion tamer.", "I am a successful store owner."]
FEW_SHOT = {
    "characteristic": ["good at math", "interested in performing", "sensitive to danger", "animal lover", "excellent business acumen"],
    "routine_habit": ["analyze financial information", "take acting classes", "do reconnaissance", "train lions", "manage inventory"],
    "goal_plan": ["to have my own audit firm", "to get auditions", "to keep his children safe", "to put on a lion show", "to open another store location"],
    "experience": ["got a degree in finance", "auditioned for a play", "discovered a security breach", "qualified as an animal trainer", "studied business management in college"],
    "relationship": ["work with clients", "sign up with a film company", "keep his friends safe", "supervised by the zoo director", "attract customers with promotions"],
}

In [ ]:
# TODO
def build_attribute_prompt(persona, relation, n=3):
    # TODO: build the chat prompt that asks the model for `n` attributes of `persona` for one `relation`
    #       (one of RELATIONS, e.g. "goal_plan"). Return a messages list for chat():
    #           [{"role": "system", "content": <one sentence telling the model to answer only with the lines>},
    #            {"role": "user",   "content": <the prompt described below>}]
    #
    # The user message mirrors the PeaCoK authors' few-shot prompt (paper Tables 12-13) and must contain, in order:
    #   1. INSTRUCTIONS[relation]                      - the task sentence for this relation
    #   2. the 5 worked examples, each written as two lines:
    #          Clue: <CLUES[i]>
    #          <REL_LABEL[relation]>: <FEW_SHOT[relation][i]>
    #      e.g. for "goal_plan":   Clue: I become an accountant.
    #                              Goal or Plan: to have my own audit firm
    #   3. the new clue:  Clue: <persona>.             - this is what the model must answer about
    #   4. a request for n different short answers (2-8 words), one per line, with no numbering and no label,
    #      about THIS person only, not a copy of the example answers (small models love to copy them).
    #
    # Hints:
    #   - zip(CLUES, FEW_SHOT[relation]) pairs each example clue with its answer; "\n\n".join(...) puts a blank
    #     line between examples.
    #   - f-strings keep this readable: f"Clue: {clue}\n{label}: {answer}".
    #   - persona.rstrip(".") avoids ending the clue with "..".

    # YOUR CODE HERE

_LABELS = {l.lower() for l in REL_LABEL.values()} | set(REL_LABEL)      # "characteristic", "routine or habit", ...
_FEW_SHOT_ANSWERS = {a.lower() for answers in FEW_SHOT.values() for a in answers}

def parse_attributes(text, n=3):
    # TODO: turn the model's raw reply `text` into a clean list of at most n attribute strings.
    #       The model does not follow the format reliably, so handle all of these (the check cell tests each one):
    #         - numbering and bullets at the start of a line:    "1. good at math", "- good at math", "* good at math"
    #         - a label before OR after the answer:               "Characteristic: good at math", "good at math: characteristic"
    #         - an echoed clue line, which is not an attribute:   "Clue: I am a baker"
    #         - several answers crammed on one line:              "baked pies, served coffee; opened a cafe"
    #         - trailing punctuation and quotes:                  'good at math.', '"good at math",'
    #         - answers copied verbatim from the few-shot examples ("to have my own audit firm")
    #         - exact duplicates and empty lines
    #       Lowercase everything, keep the order the model gave, and return out[:n].
    #
    # Hints:
    #   - Work line by line with text.splitlines().
    #   - re.sub(r"^\s*(?:[-*•]|\d+[.)])\s*", "", line) removes numbering/bullets.
    out = []

    # YOUR CODE HERE

    return out[:n]

In [ ]:
# Provided code
# Checks (do not edit)
_msgs = build_attribute_prompt("I am a baker", "goal_plan")
assert isinstance(_msgs, list) and _msgs[-1]["role"] == "user" and "I am a baker" in _msgs[-1]["content"]
assert parse_attributes("1. Characteristic: Patient with tourists.\n2. characteristic: good with knots\n- Patient with tourists", 3) == \
    ["patient with tourists", "good with knots"]
assert parse_attributes("good with knots: characteristic\nClue: I am a baker\nbakes bread daily,\nto have my own audit firm", 3) == \
    ["good with knots", "bakes bread daily"]          # suffix label, Clue line, trailing comma, copied example
print("Q A.2 checks passed")

In [ ]:
# Provided: generate a frame for each character
def generate_persona_frame(persona, n=3):
    return {rel: parse_attributes(chat(build_attribute_prompt(persona, rel, n)), n) for rel in RELATIONS}

cast_frames = {}
for name, persona in MY_CAST.items():
    set_seed(SEED)
    cast_frames[name] = generate_persona_frame(persona)
    print(f"\n== {name}: {persona}")
    for rel, attrs in cast_frames[name].items():
        print(f"  {REL_LABEL[rel]:>16}: {attrs}")


== Mara: I am a game developer who likes going to game cafes
    Characteristic: ['enjoys immersive experiences', 'fond of social gaming', 'pursues creativity in play']
  Routine or Habit: ['play games online', 'visit game cafes frequently', 'design prototypes while drinking coffee']
      Goal or Plan: ['to network and collaborate with other developers', 'to test and refine gameplay ideas', 'to explore new technologies and trends']
        Experience: ['played many video games before', 'visited several game cafes regularly', 'taught myself programming languages through online courses']
      Relationship: ['play games online', 'network with peers', 'explore gaming culture']

== Theo: I am a literature student who enjoys reading books
    Characteristic: ['enthusiastic about stories', 'proud of written words', 'passionate reader']
  Routine or Habit: ['read frequently', 'attend book clubs', 'join writing workshops']
      Goal or Plan: ['to publish her first novel', 'to attend more bo

In [ ]:
# Provided: retrieve a frame from the most similar existing PeaCoK personas (a symbolic alternative)
from sentence_transformers import SentenceTransformer, util
emb_model = SentenceTransformer(EMB_MODEL_NAME, device=DEVICE)

peacok_heads = list(facts["head"].unique())
head_emb = emb_model.encode(peacok_heads, convert_to_tensor=True, batch_size=256, show_progress_bar=True)

def retrieve_peacok_frame(persona, k_heads=3, per_rel=3):
    hits = util.semantic_search(emb_model.encode(persona, convert_to_tensor=True), head_emb, top_k=k_heads)[0]
    heads = [peacok_heads[h["corpus_id"]] for h in hits]
    sub = facts[facts["head"].isin(heads)]
    frame = {rel: sub[sub["main"] == rel]["tail"].head(per_rel).tolist() for rel in RELATIONS if rel in VALID_MAIN}
    # PeaCoK stores "relationship" as the interactivity label, not as a main relation: fill the fifth slot from it
    frame["relationship"] = sub[sub["interactivity"] == "relationship"]["tail"].head(per_rel).tolist()
    return heads, frame

for name, persona in MY_CAST.items():
    heads, frame = retrieve_peacok_frame(persona)
    print(f"\n== {name}: nearest PeaCoK personas -> {heads}")
    for rel, attrs in frame.items():
        print(f"  {REL_LABEL[rel]:>16}: {attrs}")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 90.9MB            

... (5 outputs left out)

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/15 [00:00<?, ?it/s]


== Mara: nearest PeaCoK personas -> ['i am a video game player who love to play video games', 'i am a game player who like to play video games', 'i am a video game player who like to play video games']
    Characteristic: ['lazy', 'short attention span', 'connected']
  Routine or Habit: ['stay alone', 'play video games', 'plays video games regularly']
      Goal or Plan: ['to become a champion video game player', 'to become a video game champion', 'to become a champion in a specific video game']
        Experience: ['reached the highest level of a video game', 'won a video game tournament', 'won a gaming tournament']
      Relationship: ['creates and shares gaming strategies with other players', 'shares strategies and tips with other players', 'mentors younger gamers']

== Theo: nearest PeaCoK personas -> ['i am a student who read books', 'i am a bookseller who love reading books', 'i am a novelist who love to read books']
    Characteristic: ['diligent in learning', 'inquisitive abou

## Q A.3 (2 pts): Connect your cast

Two characters' attributes form a **bridge** if they are (a) identical after normalization, (b) linked to the *same* PeaCoK attribute by `link_to_peacok` (provided), or (c) similar in meaning (cosine similarity ≥ `SOFT_THRESHOLD`).

* **Do:** implement `cast_bridges()` to find the attributes your characters share. Every character must have at least one bridge; if one doesn't, edit or regenerate its frame (use the edit cell).
* **Report:** the cast graph, the bridges for each pair, and any edits you made.

In [ ]:
# Provided code
LINK_THRESHOLD = 0.4   # min cosine to link an attribute to a PeaCoK attribute
SOFT_THRESHOLD = 0.4   # min cosine for two characters' attributes to count as a bridge

peacok_tails = list(facts["tail"].map(_norm_attr).unique())
MAX_TAILS = 80000      # lower this on CPU if encoding is slow
tail_emb = emb_model.encode(peacok_tails[:MAX_TAILS], convert_to_tensor=True, batch_size=512, show_progress_bar=True)

def link_to_peacok(attributes, threshold=LINK_THRESHOLD):
    if not attributes:
        return {}
    hits = util.semantic_search(emb_model.encode(attributes, convert_to_tensor=True), tail_emb, top_k=1)
    return {a: (peacok_tails[h[0]["corpus_id"]], round(h[0]["score"], 3)) if h and h[0]["score"] >= threshold else None
            for a, h in zip(attributes, hits)}

def all_attributes(frame):
    return [(rel, _norm_attr(a)) for rel, attrs in frame.items() for a in attrs]

Batches:   0%|          | 0/106 [00:00<?, ?it/s]

In [ ]:
# TODO
def cast_bridges(cast_frames, links, soft_threshold=SOFT_THRESHOLD):
    # TODO: find the bridges between every pair of your characters.
    #
    # Inputs
    #   cast_frames     {name: frame}; a frame is {relation: [attribute, ...]} from Q A.2 (plus any edits).
    #   links           {name: {attribute: (peacok_tail, score) or None}}, from link_to_peacok (provided):
    #                   the nearest PeaCoK attribute for each of the character's attributes, or None if no
    #                   PeaCoK attribute is close enough. Keys are the NORMALIZED attribute strings.
    #   soft_threshold  minimum cosine similarity for two attributes to count as "similar in meaning" (already given to you).
    #
    # Output
    #   {(name_a, name_b): [bridge, ...]} with one entry per UNORDERED pair of characters (3 pairs for 3
    #   characters; include a pair even if its list is empty). Each bridge is a dict
    #       {"attr_a": str, "rel_a": str, "attr_b": str, "rel_b": str, "type": str, "score": float}
    #   where attr_a/rel_a come from character a, attr_b/rel_b from character b, score is the cosine
    #   similarity of the two attributes (rounded to 3 decimals) and type is the FIRST of these that holds:
    #       "exact"             the two normalized attribute strings are identical
    #       "same_peacok_tail"  both attributes are linked and to the same PeaCoK tail
    #       "similar"           score >= soft_threshold
    #   A pair of attributes that meets none of the three is not a bridge. Sort each list by score, highest first.
    #
    # Hints
    #   - itertools.combinations(list(cast_frames), 2) gives each unordered pair of names exactly once.
    #   - Compare every attribute of a with every attribute of b (two nested loops). Encode each character's
    #     attribute list ONCE with emb_model.encode([...], convert_to_tensor=True) and get all pairwise cosines
    #     in one call.
    #   - links[name].get(attr) is None or a (tail, score) tuple, that should help with "same PeaCoK tail".
    #   - Guard against an empty attribute list (encode([]) fails): an empty frame simply has no bridges.
    #   - sorted(bridges, key=lambda d: -d["score"]) puts the strongest bridge first.
    #   - Check: a character compared with itself would give an "exact" bridge with score 1.0 for every
    #     attribute; two unrelated attributes usually score 0.1-0.4 with this embedding model.
    result = {}

    # YOUR CODE HERE

    return result

In [ ]:
# Provided code
links = {name: link_to_peacok([a for _, a in all_attributes(fr)]) for name, fr in cast_frames.items()}
bridges = cast_bridges(cast_frames, links)
for pair, bs in bridges.items():
    print(pair, len(bs), "bridges")
    for b in bs[:5]:
        print("   ", b)
lonely = [n for n in cast_frames if not any(bs for p, bs in bridges.items() if n in p)]
print("Characters with NO bridge (fix these in the edit cell, then re-run this cell):", lonely)

('Mara', 'Theo') 4 bridges
    {'attr_a': 'enjoys immersive experiences', 'rel_a': 'characteristic', 'attr_b': 'enthusiastic about stories', 'rel_b': 'characteristic', 'type': 'similar', 'score': 0.426}
    {'attr_a': 'fond of social gaming', 'rel_a': 'characteristic', 'attr_b': 'connects with fellow readers', 'rel_b': 'relationship', 'type': 'similar', 'score': 0.409}
    {'attr_a': 'pursues creativity in play', 'rel_a': 'characteristic', 'attr_b': 'enthusiastic about stories', 'rel_b': 'characteristic', 'type': 'similar', 'score': 0.405}
    {'attr_a': 'pursues creativity in play', 'rel_a': 'characteristic', 'attr_b': 'passionate reader', 'rel_b': 'characteristic', 'type': 'similar', 'score': 0.401}
('Mara', 'Ines') 5 bridges
    {'attr_a': 'visited several game cafes regularly', 'rel_a': 'experience', 'attr_b': 'to expand his cafe into multiple locations', 'rel_b': 'goal_plan', 'type': 'similar', 'score': 0.488}
    {'attr_a': 'visit game cafes frequently', 'rel_a': 'routine_habit',

# Task B: Does PeaCoK Help an LLM Stay Consistent? (9 pts)

Each PersonaChat dialogue runs twice, under two conditions:

1. **Base:** each speaker gets its ConvAI2 PersonaChat profile, about 5 persona sentences.
2. **PeaCoK:** each speaker gets the same profile plus up to 5 linked PeaCoK facts.

**Scoring rule:** both conditions are scored against the original profile only. This way, the PeaCoK speaker doesn't get credit for repeating facts the base speaker never saw.

## Setup: Generate the paired dialogues (provided)

1. **Load the data:** 40 PersonaChat dialogues that the PeaCoK authors already linked to PeaCoK facts.
2. **Run self-play:** the model plays both speakers for 12 turns under each condition. Both conditions start from the same opening line (the first line of the real dialogue, which is not scored) and use your `SEED`, so the only difference is the PeaCoK facts.
3. **Count shared facts:** how many PeaCoK attributes the two speakers share (their bridges, as in Task A).
4. **Save:** `dialogues.json`.

Self-play takes about 20–30 minutes on a free T4 GPU. On CPU, lower `N_DIALOGUES` to 20 and expect over an hour.

### To Do: upload the PersonaChat data to Colab

The next cell reads two JSON files that you download yourself and upload to a folder in Colab.

1. **Download the zip.** Open [this Google Drive link](https://drive.google.com/file/d/1uHWYb6NWcnuFQ_fAY9uOtG5WDnlUzgTv/view?usp=sharing) and click the **Download** button (the down-arrow icon at the top right). The file is too large for Google Drive to scan for viruses, so it will ask you to confirm. Click **Download anyway**. You will get `convai2_link_peacok.zip`.
2. **Unzip it on your computer.** On macOS, double-click the zip. On Windows, right-click it and choose *Extract All*. Open the `convai2_link_peacok` folder that comes out. It holds several JSON files, but you need **only these two**:
   * `convai2_personachat_original_profiles_valid.json` (about 11 MB)
   * `peacok_extend_personas_original_profiles.json` (about 102 MB)

   Do not use the `_train` or `_revised_` files, and do not rename anything.
3. **Make a new folder in Colab called `hw5_data`.** Click the folder icon (**Files**) in Colab's left sidebar. You should see `sample_data`, which means you are in `/content`. Right-click an empty spot in the Files panel, choose **New folder**, and name it exactly `hw5_data`. You can also run `!mkdir -p /content/hw5_data` in a code cell, then click the refresh icon at the top of the Files panel.
4. **Upload the two files into `hw5_data`.** Hover over `hw5_data`, click its **⋮** menu, choose **Upload**, and select both JSON files. You can also drag the files onto the folder. The 102 MB file can take a few minutes, so wait until the progress circle at the bottom of the Files panel disappears before running the next cell.
5. **Check the file structure.** It should look like this:
   ```
   /content/
   └── hw5_data/
       ├── convai2_personachat_original_profiles_valid.json
       └── peacok_extend_personas_original_profiles.json
   ```

**Note:** Colab deletes uploaded files when the runtime disconnects or is deleted (for example after a long idle period, or after *Runtime → Disconnect and delete runtime*). Restarting the session does not delete them. If the next cell says a file is missing, upload both files again.

In [ ]:
# Provided code
N_DIALOGUES = 40
N_TURNS = 12
N_EXT = 5            # PeaCoK facts given to each speaker in the PeaCoK condition

# The PeaCoK-PersonaChat data (Gao et al. 2023): the ConvAI2 dialogues and the PeaCoK facts the authors linked to
# every profile. You uploaded these two files to Colab yourself; see the instructions above this cell.
DATA_DIR = "/content/hw5_data"   # not on Colab? set this to the folder that holds the two JSON files
DIALOGUE_FILE = os.path.join(DATA_DIR, "convai2_personachat_original_profiles_valid.json")   # 1000 validation dialogues
LINKS_FILE = os.path.join(DATA_DIR, "peacok_extend_personas_original_profiles.json")        # [head, relation, tail] per profile

def load_json(path):
    try:
        with open(path, encoding="utf-8") as f:
            return json.load(f)
    except json.JSONDecodeError:
        raise ValueError(f"{path} is not valid JSON. The upload was probably cut off, so upload it again.") from None

def load_data():
    # verify: both files are in DATA_DIR, parse as JSON, and hold what we expect
    missing = [os.path.basename(p) for p in (DIALOGUE_FILE, LINKS_FILE) if not os.path.exists(p)]
    if missing:
        found = sorted(os.listdir(DATA_DIR)) if os.path.isdir(DATA_DIR) else f"no folder named {DATA_DIR}"
        raise FileNotFoundError(f"Missing {missing}. Upload both files to {DATA_DIR} as described above "
                                f"(and again if your Colab runtime was reset). Currently there: {found}")
    dialogues = load_json(DIALOGUE_FILE)
    links = load_json(LINKS_FILE)
    assert isinstance(dialogues, dict) and len(dialogues) == 1000, \
        f"expected 1000 dialogues, got {len(dialogues)}. Did you upload the *_original_profiles_valid.json file?"
    assert all(did in links for did in dialogues), \
        "some dialogues have no PeaCoK links. Did you upload peacok_extend_personas_original_profiles.json?"
    print(f"Verified: {len(dialogues)} dialogues and PeaCoK links for all of them, loaded from {DATA_DIR}")
    return dialogues, links

def link_to_statement(head, rel, tail):
    # a "_relationship" suffix on the relation is PeaCoK's interactivity label; everything else is "self"
    main, inter = (rel[:-len("_relationship")], "relationship") if rel.endswith("_relationship") else (rel, "self")
    # Paper Table 19 template, head included (same as fact_to_statement in Task A). The head is the PeaCoK persona
    # the authors matched to this speaker:
    #   "i am a circus trainer, here is what i regularly or consistently do, spend a lot of time with animals."
    desc = REL_TEXT[main] + (" " + REL_TEXT["relationship"] if inter == "relationship" else "")
    return f"{head.strip().rstrip('.')}, {desc}, {tail.strip().rstrip('.')}.".lower()

def ext_statements(facts, rng, k=N_EXT):
    # Drop exact duplicate statements (same head, relation and tail), then pick k at random.
    unique = list(dict.fromkeys(link_to_statement(*t) for t in facts))
    return rng.sample(unique, min(k, len(unique)))


raw, links = load_data()
rng = random.Random(SEED)
ids = sorted(raw)
rng.shuffle(ids)
DATA = []
for did in ids[:N_DIALOGUES]:
    s = raw[did]
    # up to N_EXT distinct linked facts per speaker, randomly chosen as the authors did, written as persona sentences
    a_ext = ext_statements(links[did]["persona1"], rng)
    b_ext = ext_statements(links[did]["persona2"], rng)
    DATA.append({"id": did, "opening": s["text"][0],
                 "A_ori": list(s["persona1_ori"]), "B_ori": list(s["persona2_ori"]), "A_ext": a_ext, "B_ext": b_ext})
print(len(DATA), "dialogues loaded. Example:")
print(json.dumps(DATA[0], indent=1)[:1200])

Verified: 1000 dialogues and PeaCoK links for all of them, loaded from /content/hw5_data
40 dialogues loaded. Example:
{
 "id": "18165",
 "opening": "hello there , how goes it ?",
 "A_ori": [
  "i've one brother.",
  "i live in alabama.",
  "i love dogs.",
  "i've three sisters.",
  "i like tacos."
 ],
 "B_ori": [
  "i'm an economics major.",
  "i'm married with two kids.",
  "i just bought a house recently.",
  "i eat a raw diet.",
  "i see the rolling stones in concert every year."
 ],
 "A_ext": [
  "i am a sad person, here is what i did in the past related to other people or social groups, lost a loved one.",
  "i am a vet, here is my character trait, have a soft spot for animals.",
  "i am a sorrowful person, here is what i did in the past, recently lost a loved one.",
  "i am a restaurant patron, here is what i regularly or consistently do, enjoy the food.",
  "i am a skilled veterinarian, here is my character trait, love of animals."
 ],
 "B_ext": [
  "i am a music lover, here is

In [ ]:
# Provided: count shared PeaCoK attributes (bridges) between the two speakers.
# Linked facts look like: "here is my character trait, love of books."
_REL_RE = re.compile(r"here is (?:my character trait|what i regularly or consistently do|what i will do or achieve in the future"
                     r"|what i did in the past)(?: related to other people or social groups)?\s*,\s*(.+)$")

def fact_tail(statement):
    s = statement.lower().strip().rstrip(".")
    m = _REL_RE.search(s)
    return _norm_attr(m.group(1) if m else s.split(",")[-1])

for d in DATA:
    d["shared_attributes"] = sorted({fact_tail(x) for x in d["A_ext"]} & {fact_tail(x) for x in d["B_ext"]})
    d["n_shared"] = len(d["shared_attributes"])
print("Dialogues whose speakers share >= 1 PeaCoK attribute:", sum(d["n_shared"] > 0 for d in DATA), "of", len(DATA))

Dialogues whose speakers share >= 1 PeaCoK attribute: 3 of 40


In [ ]:
# Provided: self-play. Each speaker sees its own lines as 'assistant' and its partner's as 'user'.
def profile_for(d, who, condition):
    return d[f"{who}_ori"] + (d[f"{who}_ext"] if condition == "peacok" else [])

def make_selfplay_messages(profile, history, me):
    system = ("You are chatting with someone you just met.\n- " + "\n- ".join(profile) +
              "\nEngange in the conversation using your traits. No narration, no name labels. "
              "These facts are about YOU, the speaker. The other person does not share them, so don't ask them about your own life; tell them about it.\n"
              "When the other person asks you something, answer from your own life before asking anything back.\n"
              "Bring up different things about yourself over the conversation instead of repeating one.\n"
              "Never say you are an AI or a language model.")
    msgs = [{"role": "system", "content": system}]
    for spk, utt in history:
        msgs.append({"role": "assistant" if spk == me else "user", "content": utt})
    return msgs

# A sentence ends at . ! or ? (plus any closing quote or bracket) followed by a space or the end of the line.
# So "5.5", "3.30pm" and "google.com" are not split, and neither are the abbreviations below.
_SENT_END = re.compile(r"[.!?]+['\")]*(?=\s|$)")
_ABBREV = {"mr", "mrs", "ms", "dr", "prof", "sr", "jr", "st", "vs", "etc", "e.g", "i.e", "a.m", "p.m", "u.s", "u.k"}

def split_sentences(text):
    # Returns (complete sentences, trailing fragment with no end mark, or "").
    sents, start = [], 0
    for m in _SENT_END.finditer(text):
        words = text[start:m.start()].split()
        at_end = not text[m.end():].strip()
        if m.group().startswith(".") and words and words[-1].lower() in _ABBREV and not at_end:
            continue                      # "Dr." / "e.g." mid-line is not the end of a sentence; at the end it is
        if text[start:m.start()].strip():  # skip punctuation with no words before it
            sents.append(text[start:m.end()].strip())
        start = m.end()
    return sents, text[start:].strip()

def clean_line(text):
    text = text.strip().split("\n")[0].strip().strip('"')
    text = re.sub(r"^(speaker\s*[ab]|a|b)\s*:\s*", "", text, flags=re.I)
    # keep at most two complete sentences and drop a trailing fragment cut off by max_new_tokens
    sents, _ = split_sentences(text)
    return " ".join(sents[:2]) if sents else text

def run_selfplay(d, condition, n_turns=N_TURNS, seed=SEED):
    set_seed(seed)
    history = [("B", d["opening"])]            # turn 0: the real opening line, not scored
    turns = []
    for t in range(1, n_turns + 1):
        me = "A" if t % 2 == 1 else "B"
        utt = clean_line(chat(make_selfplay_messages(profile_for(d, me, condition), history, me), max_new_tokens=80))
        history.append((me, utt))
        turns.append({"turn": t, "speaker": me, "utterance": utt})
    return turns

if os.path.exists("dialogues.json"):
    # Reuse dialogues saved by an earlier run (e.g. after a Colab reset). Delete the file to regenerate.
    with open("dialogues.json") as f:
        dialogues = {tuple(k.split("|")): v for k, v in json.load(f).items()}
    assert set(dialogues) == {(d["id"], c) for d in DATA for c in ["base", "peacok"]}, \
        "dialogues.json doesn't match DATA (different SEED or N_DIALOGUES?). Delete it and rerun this cell."
    print("Loaded dialogues.json (delete it to regenerate)")
else:
    dialogues = {}
    for i, d in enumerate(DATA):
        for cond in ["base", "peacok"]:
            dialogues[(d["id"], cond)] = run_selfplay(d, cond)
        if i % 5 == 0:
            print(f"{i + 1}/{len(DATA)} dialogues done")

    with open("dialogues.json", "w") as f:
        json.dump({f"{did}|{cond}": turns for (did, cond), turns in dialogues.items()}, f, indent=1)
    print("Saved dialogues.json")

d0 = DATA[0]
for cond in ["base", "peacok"]:
    print(f"\n=== {d0['id']} | {cond}\nB (opening): {d0['opening']}")
    for r in dialogues[(d0["id"], cond)]:
        print(f"{r['speaker']}: {r['utterance']}")

1/40 dialogues done
6/40 dialogues done
11/40 dialogues done
16/40 dialogues done
21/40 dialogues done
26/40 dialogues done
31/40 dialogues done
36/40 dialogues done
Saved dialogues.json

=== 18165 | base
B (opening): hello there , how goes it ?
A: Hello! I'm doing great, thank you for asking.
B: That's wonderful to hear! Do you have any hobbies or interests outside of studying?
A: Yes, actually! I do enjoy learning new languages and playing musical instruments.
B: That sounds really interesting! Have you tried any specific ones lately that you're particularly excited about?
A: I haven't had much recent time to try out new languages as my focus is on staying updated with the latest in artificial intelligence technology. However, I'm always looking forward to exploring more!
B: That makes sense. Technology is evolving rapidly these days.
A: Absolutely! It's fascinating to see how advancements in AI and machine learning impact various fields.
B: Yeah, it's amazing how technology can tran

### Download `dialogues.json`. You need to submit it with your HW.

Self-play takes 20–30 minutes, and Colab deletes your files when the runtime disconnects. Save a copy before you go on:

1. Click the folder icon (**Files**) in Colab's left sidebar. You may need to click the refresh icon at the top of the panel.
2. Find `dialogues.json` in `/content` (next to `sample_data` and `hw5_data`, not inside them).
3. Hover over it, click its **⋮** menu, and choose **Download**.

**If your runtime resets later:** upload `dialogues.json` back into `/content` (drag it onto the Files panel, not into a folder), then rerun the cells from the top. The self-play cell will load your saved file instead of generating the dialogues again. It only accepts a file made with the same `SEED` and `N_DIALOGUES`; to regenerate, delete `dialogues.json` and rerun that cell.

## Q B.1 (2 pts): Label pairs with NLI

An NLI model reads a **premise** and a **hypothesis** and labels the pair `entailment`, `neutral`, or `contradiction`.
This function is the scorer for the rest of Task B.


* **Do:** implement `nli_labels(premises, hypotheses)`, which takes two equal-length lists of strings and returns one lowercase label per pair. Read the label names from `model.config.id2label`; don't hard-code their order.

In [ ]:
# Provided code
from transformers import AutoModelForSequenceClassification
nli_tok = AutoTokenizer.from_pretrained(NLI_MODEL_NAME)
nli_model = AutoModelForSequenceClassification.from_pretrained(NLI_MODEL_NAME).to(DEVICE).eval()
print(nli_model.config.id2label)

config.json:   0%|          | 0.00/1.05k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.35k [00:00<?, ?B/s]

spm.model: reconstructing file:   0%|          |  0.00B / 2.46MB            

spm.model: downloading bytes:           |  0.00B            

tokenizer.json:   0%|          | 0.00/8.66M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

... (2 outputs left out)

model.safetensors: reconstructing file:   0%|          |  0.00B /  738MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

{0: 'contradiction', 1: 'entailment', 2: 'neutral'}


In [ ]:
# TODO
def nli_labels(premises, hypotheses, batch_size=32):
    # TODO: return a list of lowercase labels ("entailment", "neutral", or "contradiction"),
    #   one per (premises[i], hypotheses[i]) pair, in the same order as the input.
    #
    # Hints:
    #   1. Check that premises and hypotheses have the same length.
    #   2. Build a dict that maps each class id (an int) to its lowercase label name, using
    #      nli_model.config.id2label. Don't type the label order yourself.
    #   3. Loop over the pairs in chunks of batch_size: i = 0, batch_size, 2 * batch_size, ...
    #   4. For each chunk, tokenize the premises and hypotheses together:
    #        nli_tok(premises_batch, hypotheses_batch, padding=True, truncation=True, return_tensors="pt")
    #      and move the result to DEVICE.
    #   5. Run nli_model on the tokenized batch inside `with torch.no_grad():`.
    #   6. Take the argmax of the logits to get one class id per pair.

    # YOUR CODE HERE

    return out

In [ ]:
# Provided code
# Check (do not edit)
_check = nli_labels(["I have a dog.", "I have a dog.", "I have a dog."],
                    ["I don't have any pets.", "I own a dog.", "I like pizza."])
print(_check)   # note: this model often calls unrelated sentences a contradiction ("I like pizza.")
assert isinstance(_check, list) and len(_check) == 3, "Return one label per pair."
assert all(l in {"entailment", "neutral", "contradiction"} for l in _check), "Labels must be lowercase names."
assert _check[0] == "contradiction", "The first pair should be a contradiction."

# Your labels must match the model's own prediction for each pair, computed one pair at a time.
def _reference(p, h):
    enc = nli_tok(p, h, return_tensors="pt").to(nli_model.device)
    with torch.no_grad():
        logits = nli_model(**enc).logits[0]
    top2 = logits.topk(2).values
    return nli_model.config.id2label[int(logits.argmax())].lower(), float(top2[0] - top2[1])

_P = ["I have a dog.", "I have a dog.", "I have a dog.", "I have a golden retriever.",
      "I have a dog.", "I live in Paris.", "I live in France.", "I am a nurse."]
_H = ["I don't have any pets.", "I own a dog.", "I like pizza.", "I have a dog.",
      "I have a golden retriever.", "I live in France.", "I live in Paris.", "I work in a hospital."]
_ref = [_reference(p, h) for p, h in zip(_P, _H)]
_sure = [i for i, (_, margin) in enumerate(_ref) if margin > 0.5]   # skip near-ties, where padding noise could flip the label
assert len(_sure) >= 5, "Too few confident reference pairs; the check needs different pairs."
for bs in (32, 3):
    _out = nli_labels(_P, _H, batch_size=bs)
    assert isinstance(_out, list) and len(_out) == len(_P), f"With batch_size={bs}, expected {len(_P)} labels, got {len(_out)}."
    assert [_out[i] for i in _sure] == [_ref[i][0] for i in _sure], \
        f"With batch_size={bs}, labels differ from the model's prediction: check the id2label mapping, " \
        "argmax over dim=-1, premise-first order, and the batching loop."
print("Q B.1 check passed")

**Note** - NLI models trained on SNLI often label unrelated sentences as contradictions, so don't be surprised if the third pair isn't neutral in the above check.

## Q B.2 (2 pts): Measure contradictions of the profile

* **Do:** implement `persona_nli_matrix(utterances, profile)`, a matrix of NLI labels with one row per utterance and one column per original profile sentence (the profile sentence is the premise, the utterance is the hypothesis). Then implement `contradiction_rate(M)`: the fraction of utterances (rows) that contradict at least one profile sentence.
* **Report:** the mean contradiction rate for base and for PeaCoK.

In [ ]:
# TODO
def persona_nli_matrix(utterances, profile):
    # TODO: return a list of lists of NLI labels, one row per utterance and one column per profile sentence,
    #   so M[i][j] = nli label for (premise = profile[j], hypothesis = utterances[i]).
    #
    # Hints:
    #   1. Build two flat lists with one entry per (utterance, profile sentence) pair, going utterance by utterance:
    #        premises:   profile[0], profile[1], ..., profile[-1], profile[0], profile[1], ...
    #        hypotheses: utterances[0] repeated len(profile) times, then utterances[1] repeated, ...
    #   2. Call nli_labels ONCE on the two flat lists (it batches for you, so this is much faster than one call per pair).
    #   3. Cut the flat list of labels into consecutive chunks of len(profile): each chunk is one row.

     # YOUR CODE HERE

def contradiction_rate(M):
    # TODO: return the fraction of rows (utterances) that contradict AT LEAST ONE profile sentence.
    #
    # Hints:
    #   1. If M is empty, return float("nan"): there is nothing to measure.
    #   2. Count the rows that contain "contradiction" anywhere.

     # YOUR CODE HERE

assert contradiction_rate([["neutral", "contradiction"], ["entailment", "neutral"]]) == 0.5

In [ ]:
# Provided code
# Check (do not edit)
_utts = ["I don't have any pets.", "I own a dog.", "I have a golden retriever."]
_prof = ["I have a dog.", "I live in Paris."]
_M = persona_nli_matrix(_utts, _prof)
assert isinstance(_M, list) and len(_M) == len(_utts) and all(isinstance(r, list) and len(r) == len(_prof) for r in _M), \
    "Rows must be utterances and columns profile sentences: expected a 3 x 2 list of lists."
_expected = nli_labels([p for _ in _utts for p in _prof], [u for u in _utts for _ in _prof])
assert [l for row in _M for l in row] == _expected, \
    "Cell [i][j] must be NLI(premise = profile[j], hypothesis = utterances[i])."
assert _M[0][0] == "contradiction", "'I have a dog.' -> 'I don't have any pets.' should be a contradiction."
assert persona_nli_matrix([], _prof) == [], "No utterances should give an empty matrix."

assert contradiction_rate([["neutral", "contradiction"], ["entailment", "neutral"]]) == 0.5
assert contradiction_rate([["contradiction", "contradiction"], ["neutral", "entailment"], ["neutral", "neutral"]]) == 1 / 3, \
    "Count rows with at least one contradiction, divided by the number of rows."
assert contradiction_rate([["neutral", "contradiction", "neutral"]]) == 1.0, \
    "A row counts once if it has any contradiction."
assert np.isnan(contradiction_rate([])), "Return float('nan') for an empty matrix."
print("Q B.2 check passed")

In [ ]:
# Provided: one NLI matrix per speaker per dialogue, always against the ORIGINAL profile
def lines_of(turns, who):
    return [r["utterance"] for r in turns if r["speaker"] == who]

MATRICES = {}
for d in DATA:
    for cond in ["base", "peacok"]:
        for who in ["A", "B"]:
            MATRICES[(d["id"], cond, who)] = persona_nli_matrix(lines_of(dialogues[(d["id"], cond)], who), d[f"{who}_ori"])

def per_dialogue(metric_fn, cond, matrices=None):
    # one value per dialogue = mean over its two speakers (uses MATRICES unless you pass other matrices)
    matrices = MATRICES if matrices is None else matrices
    return np.array([np.nanmean([metric_fn(matrices[(d["id"], cond, w)]) for w in "AB"]) for d in DATA])


for cond in ["base", "peacok"]:
    print(f"{cond:>7}: mean contradiction rate = {np.nanmean(per_dialogue(contradiction_rate, cond)):.3f}")

   base: mean contradiction rate = 0.269
 peacok: mean contradiction rate = 0.260


## Q B.3 (1 pt): Measure persona coverage

A speaker can avoid contradictions by never talking about itself. Coverage catches that: it measures how much of the profile the speaker actually expresses.

* **Do:** implement `persona_coverage(M)`, which returns the fraction of profile sentences (columns) entailed by at least one utterance.
* **Report:** the mean coverage for base and for PeaCoK.



In [ ]:
# TODO
def persona_coverage(M):
    # TODO: M[i][j] is the NLI label for (premise = utterance i, hypothesis = profile sentence j).
    #   Return the fraction of columns (profile sentences) with at least one "entailment".
    #   Return float("nan") if M is empty or has no columns: there is nothing to measure.

    # YOUR CODE HERE

In [ ]:
# Provided code
# Check (do not edit)
assert persona_coverage([["entailment", "neutral"], ["neutral", "neutral"]]) == 0.5
assert persona_coverage([["entailment", "neutral"], ["entailment", "neutral"], ["neutral", "neutral"]]) == 0.5, \
    "A profile sentence counts once, however many utterances entail it."
assert persona_coverage([["neutral", "neutral", "neutral"], ["entailment", "entailment", "neutral"]]) == 2 / 3, \
    "Divide by the number of profile sentences (columns), not utterances (rows)."
assert persona_coverage([["neutral", "neutral"], ["neutral", "entailment"]]) == 0.5, \
    "Check every row, not just the first."
assert persona_coverage([["contradiction", "neutral"], ["neutral", "contradiction"]]) == 0.0, \
    "Only 'entailment' counts as covering a profile sentence."
assert np.isnan(persona_coverage([])) and np.isnan(persona_coverage([[], []])), \
    "Return float('nan') when there is nothing to measure."
print("Q B.3 check passed")

In [ ]:
# Provided: coverage matrices. M[i][j] = NLI(premise = utterance i, hypothesis = profile sentence j):
# does something the speaker SAID imply this profile sentence? This is the reverse of the Q B.2 matrix.
def coverage_matrix(utterances, profile):
    prem = [u for u in utterances for _ in profile]
    hyp = [p for _ in utterances for p in profile]
    flat = nli_labels(prem, hyp) if prem else []
    return [flat[i * len(profile):(i + 1) * len(profile)] for i in range(len(utterances))]

COVER_MATRICES = {}
for d in DATA:
    for cond in ["base", "peacok"]:
        for who in ["A", "B"]:
            COVER_MATRICES[(d["id"], cond, who)] = coverage_matrix(lines_of(dialogues[(d["id"], cond)], who), d[f"{who}_ori"])

for cond in ["base", "peacok"]:
    print(f"{cond:>7}: mean coverage = {np.nanmean(per_dialogue(persona_coverage, cond, COVER_MATRICES)):.3f}")

   base: mean coverage = 0.071
 peacok: mean coverage = 0.077


## Q B.4 (4 pts): Decide whether PeaCoK helps

Two metrics decide whether PeaCoK helps the LLM stay consistent: the **contradiction rate** from B.2 and the **persona coverage** from B.3. B.1 provides the NLI scorer that both metrics use; it is not a metric itself.

* **Do:** run the provided results cell. It compares base and PeaCoK on both metrics for all dialogues, then separately for dialogues whose speakers share no PeaCoK attribute and for dialogues whose speakers share at least one.

* **Report (1pt):** the table.

* **Answer (1-2 sentences) (3pts):**
  1. Does PeaCoK make the LLM more consistent? Use the Verdict in the two "all dialogues" rows.
  2. The paper found a larger gain when the speakers share attributes. Do you see that here? Check N for the ">= 1 shared attribute" rows to draw a conclusion.
  3. Is either condition consistent only because the speaker says little about itself? Compare its contradiction rate with its coverage.

### How to read the table

| Column | What it means |
|---|---|
| **Metric** | Contradiction Rate (lower is better) and Coverage (higher is better) |
| **Dialogues** | The dialogues used in this row (more details below). |
| **N** | The number of dialogues in this row. |
| **Base** | The mean of the metric over the N dialogues when each speaker sees only its original profile. |
| **PeaCoK** | The mean of the metric over the same N dialogues when each speaker sees its original profile plus up to 5 PeaCoK facts. |
| **Difference** | PeaCoK − Base. For contradiction rate, a **negative** difference means PeaCoK contradicts less, which is better. For coverage, a **positive** difference means PeaCoK expresses more of the profile, which is better. |
| **Verdict** | Whether the difference is larger than what chance alone would produce (see below). |

**Metric**
1) B.2 - Contradiction Rate (lower is better): the fraction of a speaker's utterances that contradict at least one of its original profile sentences.
2) B.3 - Coverage (higher is better): the fraction of the original profile sentences that the speaker's utterances entail.

**Dialogues:**
1) *all dialogues* -  every dialogue.
2) *no shared attributes* - the two speakers' PeaCoK facts have no attribute in common.
3) *>= 1 shared attribute* -  they have at least one.

**How the verdict is decided?** \
Every dialogue runs under both conditions with the same opening line and seed, so each dialogue gives one paired difference (PeaCoK − Base). With only N dialogues, the average of these differences can be nonzero by luck. To check, the cell resamples the N dialogues, recomputes the average difference each time, and keeps the middle 95% of those averages. This range is a 95% confidence interval.


* **no real difference:** Don't call the Difference a gain or a loss, however large it looks. This does not prove that PeaCoK has no effect; it means these N dialogues cannot tell.
* **too few dialogues:** N < 5, which is too few to decide.

*Example:* a contradiction-rate row with Difference = +0.04 and Verdict = *no real difference* means the PeaCoK speakers contradicted their profiles slightly more often on average, but the gap is within normal variation between dialogues.

**Read the two metrics together.** A speaker that never talks about itself cannot contradict its profile. So a lower contradiction rate is good news only if coverage does not drop at the same time.


In [ ]:
# Provided: paired comparison of PeaCoK vs. base. The verdict uses a 95% bootstrap confidence interval on the
# per-dialogue difference (PeaCoK - base). The interval is computed here but not shown in the table.
def paired_compare(base, peacok, n_boot=2000, seed=SEED):
    base, peacok = np.asarray(base, float), np.asarray(peacok, float)
    keep = ~(np.isnan(base) | np.isnan(peacok))
    diff = peacok[keep] - base[keep]
    if len(diff) == 0:
        return {"n": 0, "base": np.nan, "peacok": np.nan, "diff": np.nan, "ci_low": np.nan, "ci_high": np.nan}
    rng = np.random.default_rng(seed)
    boots = [rng.choice(diff, len(diff)).mean() for _ in range(n_boot)]
    return {"n": int(len(diff)), "base": base[keep].mean(), "peacok": peacok[keep].mean(), "diff": diff.mean(),
            "ci_low": np.percentile(boots, 2.5), "ci_high": np.percentile(boots, 97.5)}

# metric -> (one value per dialogue for a condition, which direction is better)
METRICS = {
    "B.2 contradiction rate (lower is better)": (lambda cond: per_dialogue(contradiction_rate, cond), "lower"),
    "B.3 coverage (higher is better)":          (lambda cond: per_dialogue(persona_coverage, cond, COVER_MATRICES), "higher"),
}
shared = np.array([d["n_shared"] > 0 for d in DATA])
GROUPS = [("all dialogues", np.ones(len(DATA), dtype=bool)), ("no shared attributes", ~shared), (">= 1 shared attribute", shared)]

rows = []
for name, (fn, better) in METRICS.items():
    b, p = fn("base"), fn("peacok")
    for group, mask in GROUPS:
        rows.append({"metric": name, "better if": better, "dialogues": group, **paired_compare(b[mask], p[mask])})
results = pd.DataFrame(rows)

def verdict(r):
    # a difference is real only if its 95% confidence interval excludes 0
    if r["n"] < 5:
        return "too few dialogues"
    if r["ci_low"] <= 0 <= r["ci_high"]:
        return "no real difference"
    peacok_lower = r["ci_high"] < 0
    return "PeaCoK better" if peacok_lower == (r["better if"] == "lower") else "PeaCoK worse"

results["verdict"] = results.apply(verdict, axis=1)
TABLE_COLUMNS = {"metric": "Metric", "dialogues": "Dialogues", "n": "N", "base": "Base",
                 "peacok": "PeaCoK", "diff": "Difference", "verdict": "Verdict"}
display(results[list(TABLE_COLUMNS)].rename(columns=TABLE_COLUMNS).round(3))
if shared.sum() < 5 or (~shared).sum() < 5:
    print("Note: one group has fewer than 5 dialogues, so it can't support a conclusion.")

,Metric,Dialogues,N,Base,PeaCoK,Difference,Verdict
0,Q2.2 contradiction rate (lower is better),all dialogues,40,0.269,0.260,-0.008,no real difference
1,Q2.2 contradiction rate (lower is better),no shared attributes,37,0.264,0.264,-0.000,no real difference
2,Q2.2 contradiction rate (lower is better),>= 1 shared attribute,3,0.333,0.222,-0.111,too few dialogues
3,Q2.3 coverage (higher is better),all dialogues,40,0.071,0.078,0.007,no real difference
4,Q2.3 coverage (higher is better),no shared attributes,37,0.064,0.078,0.014,no real difference
5,Q2.3 coverage (higher is better),>= 1 shared attribute,3,0.150,0.067,-0.083,too few dialogues


Note: one group has fewer than 5 dialogues, so it can't support a conclusion.


## `You did it! This is the end of HW 5!!`